# 05. Agent Development Frameworks: Professional Implementations

This lab explores how modern agent frameworks encapsulate runtime mechanics. We will move beyond toy examples and examine professional practices: **dependency injection, strict typing, persistent checkpointers, and durable execution.**

**Core Rule:** The framework is NOT the architecture. We must define the domain models and tool boundaries *first*, and then port the identical business logic to different runtimes to evaluate them objectively.

## Part 1 — Provider-Neutral Domain Models & Shared Tools
We define strict Pydantic inputs and outputs. This ensures our business logic (the tools) remains totally uncoupled from whether we use LangGraph, PydanticAI, or a raw loop.

In [1]:
from pydantic import BaseModel, Field
import json
from typing import List, Optional, Any
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('Northstar')

# 1. Application-Owned Tools
class HealthRequest(BaseModel):
    region: str

class DeploymentRequest(BaseModel):
    service: str

def get_service_health(req: HealthRequest) -> str:
    logger.info(f'[TOOL] get_service_health({req.region})')
    return json.dumps({'region': req.region, 'status': 'DEGRADED', 'error_rate': '15%'})

def get_recent_deployments(req: DeploymentRequest) -> str:
    logger.info(f'[TOOL] get_recent_deployments({req.service})')
    return json.dumps({'service': req.service, 'latest_deployment_id': 'dep_eu_114', 'time': '2 hours ago'})

# 2. Provider-Neutral Internal Runtime Models
class ToolCall(BaseModel):
    id: str
    name: str
    arguments_json: str

class ModelDecision(BaseModel):
    tool_calls: List[ToolCall] = Field(default_factory=list)
    final_answer: Optional[str] = None

print('Domain objects and bounded tools successfully initialized.')

Domain objects and bounded tools successfully initialized.


## Part 2 — The Framework-Neutral Baseline (Raw Loop)
To understand what frameworks actually do, we first build the loop manually. Our `MockLLM` returns our internal `ModelDecision` object (not a provider-specific JSON shape), proving we own the loop.

In [2]:
class MockLLM:
    def __init__(self):
        self.turns = 0
    def chat(self, memory: list) -> ModelDecision:
        self.turns += 1
        if self.turns == 1:
            return ModelDecision(
                tool_calls=[
                    ToolCall(id='tc_1', name='get_service_health', arguments_json='{"region": "EU"}')
                ]
            )
        return ModelDecision(final_answer='The EU region is degraded due to deployment dep_eu_114. Rollback required.')

def raw_agent_loop(query: str):
    llm = MockLLM()
    memory = [query]
    
    while True:
        decision = llm.chat(memory)
        memory.append(decision)
        
        if decision.final_answer:
            return decision.final_answer # Terminal state
            
        for tc in decision.tool_calls:
            # Application completely owns routing and execution
            if tc.name == 'get_service_health':
                res = get_service_health(HealthRequest.model_validate_json(tc.arguments_json))
            elif tc.name == 'get_recent_deployments':
                res = get_recent_deployments(DeploymentRequest.model_validate_json(tc.arguments_json))
            else:
                res = 'Error: Unknown tool'
            memory.append({'tool_id': tc.id, 'result': res})

final_output = raw_agent_loop('Checkout failures in EU.')
print('\nRaw Baseline Final Output:', final_output)

INFO: [TOOL] get_service_health(EU)



Raw Baseline Final Output: The EU region is degraded due to deployment dep_eu_114. Rollback required.


## Part 3 — PydanticAI: Typed Outputs & Dependency Injection
PydanticAI abstracts the loop away, but its primary value is **typing**. It enforces validated outputs (no arbitrary confidence scores without evidence) and cleanly injects dependencies via `RunContext`.

In [3]:
import os
try:
    from pydantic_ai import Agent, RunContext
    from dataclasses import dataclass

    @dataclass
    class AppDependencies:
        db_connection_str: str
        environment: str

    class FinalDecision(BaseModel):
        requires_rollback: bool
        target_deployment: str
        evidence_ids: List[str] = Field(..., description='IDs of logs proving the issue')
        rationale_summary: str

    # Agent enforces that final result is a validated FinalDecision
    typed_agent = Agent('openai:gpt-4o-mini', deps_type=AppDependencies, result_type=FinalDecision)

    @typed_agent.tool
    def fetch_health(ctx: RunContext[AppDependencies], region: str) -> str:
        # Clean dependency injection (e.g. using ctx.deps.db_connection_str)
        return get_service_health(HealthRequest(region=region))

    print('PydanticAI architecture successfully defined.')
    
    if os.getenv('OPENAI_API_KEY'):
        deps = AppDependencies(db_connection_str='postgres://app', environment='prod')
        result = typed_agent.run_sync('Investigate EU checkout failures.', deps=deps)
        print('\nPydanticAI Typed Output:\n', result.data)
except ImportError:
    print('To run this framework, `pip install pydantic-ai`')


To run this framework, `pip install pydantic-ai`


## Part 4 — LangGraph: Durable Execution & HITL
LangGraph models the agent as a state machine. It offers `MemorySaver` (or SqliteSaver) to durably persist the thread. A graph interrupted at a breakpoint can be safely resumed later.

In [4]:
try:
    from typing import Annotated
    from typing_extensions import TypedDict
    from langgraph.graph import StateGraph, START, END
    from langgraph.graph.message import add_messages
    from langgraph.checkpoint.memory import MemorySaver

    class GraphState(TypedDict):
        messages: Annotated[list, add_messages]
        human_approved: bool

    def agent_node(state: GraphState):
        logger.info('Node: Agent evaluating state.')
        return {'messages': [{'role': 'assistant', 'content': 'Proposing rollback for dep_eu_114.'}]}

    def review_node(state: GraphState):
        logger.info('Node: Human review completed. Action executed.')
        return {'human_approved': True}

    builder = StateGraph(GraphState)
    builder.add_node('agent', agent_node)
    builder.add_node('review', review_node)
    builder.add_edge(START, 'agent')
    builder.add_edge('agent', 'review')
    builder.add_edge('review', END)

    # Real checkpointer ensures persistence between asynchronous steps
    checkpointer = MemorySaver()
    graph = builder.compile(checkpointer=checkpointer, interrupt_before=['review'])
    
    config = {'configurable': {'thread_id': 'incident_101'}}
    
    print('\n--- 1. Initial Run (Hits Interrupt) ---')
    for event in graph.stream({'messages': [{'role': 'user', 'content': 'Fix EU.'}]}, config):
        pass
    
    state = graph.get_state(config)
    print('Graph is paused at node:', state.next)
    
    print('\n--- 2. Resuming execution after Human Approval ---')
    # We resume the exact thread_id without re-evaluating the agent node
    for event in graph.stream(None, config):
        pass
    print('Final State:', graph.get_state(config).values['human_approved'])
    
except ImportError:
    print('To run this framework, `pip install langgraph langgraph-checkpoint`')


INFO: Node: Agent evaluating state.


INFO: Node: Human review completed. Action executed.



--- 1. Initial Run (Hits Interrupt) ---
Graph is paused at node: ('review',)

--- 2. Resuming execution after Human Approval ---
Final State: True


## Part 5 — OpenAI Raw SDK Implementation
If you want full loop ownership without a heavy framework, you can use the official `openai` SDK, keeping application routing in standard Python code.

In [5]:
if os.getenv('OPENAI_API_KEY'):
    try:
        from openai import OpenAI
        client = OpenAI()
        
        real_tools = [
            {'type': 'function', 'function': {'name': 'get_service_health', 'description': 'Check health for a region', 'parameters': HealthRequest.model_json_schema()}},
        ]
        
        messages = [{'role': 'user', 'content': 'Checkout failures in EU. Investigate.'}]
        print('\n--- OpenAI Raw Loop Execution ---')
        
        response = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=real_tools)
        msg = response.choices[0].message
        
        if msg.tool_calls:
            logger.info(f'Model requested tool: {msg.tool_calls[0].function.name}')
            # App executes tool
            res = get_service_health(HealthRequest.model_validate_json(msg.tool_calls[0].function.arguments))
            messages.append(msg)
            messages.append({'role': 'tool', 'tool_call_id': msg.tool_calls[0].id, 'content': res})
            
            # Terminal step
            final_res = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
            print('Final Output:', final_res.choices[0].message.content)
    except Exception as e:
        print('Error running OpenAI SDK:', e)
else:
    print('Skipping real execution. Set OPENAI_API_KEY to run.')

Skipping real execution. Set OPENAI_API_KEY to run.


## Part 6 — Empirical Framework Comparison
We can objectively compare the value of these runtimes using a shared rubric.

In [6]:
import pandas as pd

comparison = [
    {'Framework': 'Raw Baseline', 'Loop Ownership': 'App', 'Output Typing': 'Manual', 'Persistence': 'None (In-Memory)', 'Complexity': 'Low'},
    {'Framework': 'OpenAI SDK', 'Loop Ownership': 'App', 'Output Typing': 'via .parse()', 'Persistence': 'None (In-Memory)', 'Complexity': 'Low'},
    {'Framework': 'PydanticAI', 'Loop Ownership': 'Framework', 'Output Typing': 'Validated Pydantic', 'Persistence': 'Optional', 'Complexity': 'Medium'},
    {'Framework': 'LangGraph', 'Loop Ownership': 'Framework (Graph)', 'Output Typing': 'Manual', 'Persistence': 'Durable (Checkpointers)', 'Complexity': 'High'}
]
display(pd.DataFrame(comparison))

,Framework,Loop Ownership,Output Typing,Persistence,Complexity
0,Raw Baseline,App,Manual,None (In-Memory),Low
1,OpenAI SDK,App,via .parse(),None (In-Memory),Low
2,PydanticAI,Framework,Validated Pydantic,Optional,Medium
3,LangGraph,Framework (Graph),Manual,Durable (Checkpointers),High
